# Lesson 03 — Part 1 (Part A): Connect, first call, settings, streaming, structured output
**Course:** GenAI Basics | **Instructor:** Igor Rubanovich (EPAM Systems) | **Model:** `gemini-3.5-flash-lite`

This is Part A of `L03_First_API_Call_and_Prompt_Patterns.ipynb` (Steps 0–1.4).

---
# Part A — Connect to the model

## What is an API?
An **API** (Application Programming Interface) is a "door" through which one program asks another program to do work.
Gemini lives on Google's servers. Your Python code doesn't run the model — it sends an **HTTP request** over the internet and gets a **response** back.

```
  YOUR CODE (Colab)                                   GOOGLE (Gemini servers)
 ┌────────────────────┐   ① HTTPS request            ┌─────────────────────────┐
 │ client.models      │ ───────────────────────────▶ │  checks API key 🔑       │
 │  .generate_content │   • API key  (who you are)   │  runs the model         │
 │  (model, prompt)   │   • model    (which brain)   │  counts tokens          │
 │                    │   • prompt   (your question) │                         │
 │                    │ ◀─────────────────────────── │                         │
 │ response.text      │   ② HTTPS response (JSON)    │                         │
 │ response.usage...  │   • text   (the answer)      └─────────────────────────┘
 └────────────────────┘   • tokens (the "receipt")
```

**Analogy:** a restaurant. The **menu** is the API documentation, your **order** is the request, the **kitchen** is the model, the **dish + bill** is the response. The **API key** is your membership card: it tells the restaurant who pays.

> 🔐 **An API key is like a password.** Anyone who has it can spend your quota. Never paste it into code, screenshots, chat or GitHub.

## Step 0: Create your Gemini API key (≈ 2 minutes, free)

**0.1 — Create the key in Google AI Studio**
1. Open **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)** and sign in with a Google account (a personal Gmail works).
2. Accept the terms if asked.
3. Click **Create API key**. If asked for a project, pick the default one or create a new project.
4. Click **Copy** next to the new key. Keep this browser tab open — you'll paste the key in a moment.

**0.2 — Store the key in Colab Secrets (not in code!)**
1. In this Colab notebook, click the **🔑 key icon ("Secrets")** in the left sidebar.
2. Click **+ Add new secret**.
3. **Name:** `GEMINI_API_KEY` (exactly like this, upper-case). **Value:** paste your key.
4. Switch on the **Notebook access** toggle.

```
 AI Studio  ──copy key──▶  Colab 🔑 Secrets  ──userdata.get()──▶  genai.Client  ──▶  Gemini
                           (stored safely,                        (adds the key to
                            never in the code)                     every request)
```

**0.3 — Run the cell below.** It installs Google's official Python library (SDK) `google-genai` and creates a `client` — the object that will send all our requests.
If Colab asks *"Grant access to GEMINI_API_KEY?"* — click **Grant access**.

> 💡 **Free tier limits.** The free key has a per-minute and per-day request limit. If you see error `429 RESOURCE_EXHAUSTED`, wait ~1 minute and re-run the cell.

In [16]:
# Install Google's official GenAI SDK (takes ~10 seconds)
!pip install -q -U google-genai

from google import genai
from google.genai import types
from google.colab import userdata

MODEL = "gemini-3.5-flash-lite"   # one place to change the model for the whole notebook

try:
    api_key = userdata.get("GEMINI_API_KEY")   # read the key from Colab Secrets
except Exception as e:
    raise RuntimeError(
        " Could not read GEMINI_API_KEY from Colab Secrets.\n"
        "   Check: Secrets → name is exactly GEMINI_API_KEY → 'Notebook access' is ON."
    ) from e

# The free tier is sometimes busy (errors 429 / 503) → let the SDK retry automatically.
# Note: google-genai does NOT retry by default — retries are switched on here, once, for every call.
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        timeout=60_000,                                                    # give up on a hung request after 60 s (milliseconds)
        retry_options=types.HttpRetryOptions(attempts=5, initial_delay=2),  # retry 408 / 429 / 5xx with growing pauses
    ),
)
print(f"✅ Client ready. Model for this lesson: {MODEL}")

✅ Client ready. Model for this lesson: gemini-3.5-flash-lite


## Step 1: First API call & the token "receipt"

The whole call is **one function** with two required arguments:

```python
response = client.models.generate_content(
    model    = MODEL,                # WHICH model answers
    contents = "your prompt here",   # WHAT you ask
)
response.text            # the answer as a string
response.usage_metadata  # the receipt: how many tokens were used
```

**Tokens** are the units LLMs read, write and **bill** — pieces of words, roughly ¾ of an English word on average:

```
"Tokenization is fun!"  ──▶  [Token][ization][ is][ fun][!]   ≈ 5 tokens
```

The receipt has three kinds of tokens:
* **Prompt tokens** — what you sent (input).
* **Output tokens** — the visible answer.
* **Thinking tokens** — Gemini 3.x models "think" silently before answering. You don't see these tokens, but **you pay for them as output**.

In [17]:
import time

start = time.time()
response = client.models.generate_content(
    model=MODEL,
    contents="What is an API? Answer in one simple sentence for a beginner.",
)
latency = time.time() - start

print("=== MODEL RESPONSE ===")
print(response.text.strip())

usage = response.usage_metadata
print("\n=== TOKEN RECEIPT ===")
print(f"Prompt tokens (input):   {usage.prompt_token_count}")
print(f"Output tokens (answer):  {usage.candidates_token_count}")
print(f"Thinking tokens:         {usage.thoughts_token_count or 0}")
print(f"Total tokens:            {usage.total_token_count}")
print(f"Latency:                 {latency:.2f} s")

=== MODEL RESPONSE ===
An API is like a messenger that takes your request to a system, brings back the answer, and delivers it to you.

=== TOKEN RECEIPT ===
Prompt tokens (input):   15
Output tokens (answer):  25
Thinking tokens:         0
Total tokens:            40
Latency:                 0.73 s


### 🔍 Under the hood: it's just HTTP (optional, but eye-opening)
The SDK is a convenience wrapper. Below is **the same call** made with a plain HTTP request using the `requests` library — no Google SDK at all.
Notice the three ingredients from the diagram: **URL with the model name**, **API key in a header**, **prompt in a JSON body**. Any language that can send HTTP (JavaScript, Java, C#, `curl`…) can talk to Gemini this way.

In [18]:
import requests, json, time

url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"
headers = {"x-goog-api-key": api_key, "Content-Type": "application/json"}
body = {"contents": [{"parts": [{"text": "Say hello to the workshop in 5 words."}]}]}

for attempt in range(3):                       # simple retry: the free tier is sometimes busy
    http_response = requests.post(url, headers=headers, json=body, timeout=60)
    if http_response.status_code not in (429, 503):
        break
    print(f"⏳ Server busy ({http_response.status_code}), retrying in 10 s…")
    time.sleep(10)

print("HTTP status:", http_response.status_code)   # 200 = OK, 4xx = your mistake, 5xx = server problem
http_response.raise_for_status()

data = http_response.json()
print("\n=== Answer, dug out of the raw JSON ===")
print(data["candidates"][0]["content"]["parts"][0]["text"])
print("\n=== Raw usageMetadata (the receipt) ===")
print(json.dumps(data.get("usageMetadata", {}), indent=2))

HTTP status: 200

=== Answer, dug out of the raw JSON ===
Hello workshop, welcome to creation.

=== Raw usageMetadata (the receipt) ===
{
  "promptTokenCount": 10,
  "candidatesTokenCount": 7,
  "totalTokenCount": 17,
  "promptTokensDetails": [
    {
      "modality": "TEXT",
      "tokenCount": 10
    }
  ],
  "serviceTier": "standard"
}


### A tiny helper for the rest of the lesson
From now on we only change **the prompt**, so we wrap the call into a helper `ask()`.
It also lets us set options we'll use later: a **system instruction**, the **thinking level** and the **temperature**.

In [19]:
import json, re, time

def ask(prompt, system=None, thinking="minimal", temperature=None):
    """Send one prompt to Gemini and return the response object.

    prompt   – the text you send (user message)
    system   – optional system instruction (the model's role / standing rules)
    thinking – how much the model may 'think' silently: minimal | low | medium | high
    temperature – randomness of the answer: 0.0 = repeatable, 1.0+ = creative (None = model default)
    """
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        thinking_config=types.ThinkingConfig(thinking_level=thinking),
    )
    return client.models.generate_content(model=MODEL, contents=prompt, config=config)

print(ask("Reply with exactly: helper works").text)

helper works


## Step 1.1: Tune the model — `temperature` and `max_output_tokens`

Every call can carry settings in `types.GenerateContentConfig`. Two you will use all the time:

| Setting | What it does | Typical value |
|---|---|---|
| `temperature` | How "adventurous" the choice of the next token is | **`0.0`** for extraction, JSON, labels (repeatable); **`0.7–1.0+`** for names, ideas, creative text |
| `max_output_tokens` | A hard cap on the answer length — and on the cost | When the cap is hit, `finish_reason` = **`MAX_TOKENS`** |

```
 next token after "Robo…":   Brew 0.62 | Bean 0.21 | Byte 0.11 | Gear 0.06
 temperature 0.0  ──▶ always "Brew"            (same answer every run)
 temperature 1.5  ──▶ sometimes Bean / Byte…   (different answer every run)
```

In [ ]:
# 1.1a temperature: the same creative prompt, 3 runs at 0.0 and 3 runs at 1.5
creative_prompt = "Invent a name for a coffee shop run by robots. Reply with the name only."

for temp in [0.0, 1.5]:
    print(f"=== temperature={temp} (3 runs) ===")
    for run in range(3):
        print(f"  run {run + 1}: {ask(creative_prompt, temperature=temp).text.strip()}")
    print()
# Expected: at 0.0 the runs are (almost) identical; at 1.5 the names differ.

In [ ]:
# 1.1b max_output_tokens: a hard cap on length (and cost)
short = client.models.generate_content(
    model=MODEL,
    contents="Explain how HTTPS works in 300 words.",
    config=types.GenerateContentConfig(
        max_output_tokens=40,
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    ),
)
print(short.text or "(no visible text: the limit was spent before the answer started)")
print("\nFinish reason:", short.candidates[0].finish_reason)   # MAX_TOKENS = the answer was cut by our cap
print("Output tokens:", short.usage_metadata.candidates_token_count)

## Step 1.2: Streaming — show the first words immediately

A regular call returns **nothing** until the whole answer is ready. `generate_content_stream` keeps the connection open and sends the answer in **chunks** while the model is still writing (Server-Sent Events).

```
 generate_content          ░░░░░░░░░░░░░░░░░░░░░░░░░░░░▶ [whole text]      (user stares at an empty screen)
 generate_content_stream   ░░▶[chunk][chunk][chunk][chunk][chunk]           (first words almost at once)
                             ▲ TTFT = time to first token
```

Total time is about the same — only the **waiting feels shorter**. Stream for humans (chat UIs); use a normal call for background jobs and JSON.

In [ ]:
# 1.2 Same prompt twice: blocking call vs streaming, measuring time to the first character
stream_prompt = "Explain in 8 short sentences why an LLM API bills by tokens, not by words."
fast = types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_level="minimal"))

t0 = time.time()
blocking = client.models.generate_content(model=MODEL, contents=stream_prompt, config=fast)
blocking_time = time.time() - t0
print(f"=== 1.2a BLOCKING: first character after {blocking_time:.2f} s ===")
print(blocking.text.strip())

print("\n=== 1.2b STREAMING ===")
t0, ttft, last = time.time(), None, None
for chunk in client.models.generate_content_stream(model=MODEL, contents=stream_prompt, config=fast):
    if ttft is None:
        ttft = time.time() - t0                    # time to first token
    print(chunk.text or "", end="", flush=True)    # flush=True → print each chunk immediately
    last = chunk
total = time.time() - t0

print(f"\n\nBlocking: first character after {blocking_time:.2f} s")
print(f"Streaming: first character after {ttft:.2f} s, full answer after {total:.2f} s")
if last is not None and last.usage_metadata:
    print("Tokens (reported in the last chunk):", last.usage_metadata.total_token_count)

## Step 1.3: Structured Output — JSON guaranteed by the API

Asking *"reply with JSON"* in the prompt is a **wish**: the model may add ```` ```json ````, a polite sentence, or invent its own keys (you'll see this in Step 2).
**Structured Output** is a **contract**: you describe the fields as a Python class (Pydantic) and pass it to the API. The server only lets the model produce tokens that fit the schema.

```
 email text ──▶ response_schema=TicketTriage ──▶ server masks tokens outside the schema ──▶ valid JSON ──▶ .parsed (Python object)
```

In [ ]:
from typing import Literal
from pydantic import BaseModel

# 1.3a The data contract: exact fields, types and allowed values
class TicketTriage(BaseModel):
    category: Literal["billing", "technical", "general"]
    urgent: bool
    amount_usd: float | None   # None if the email mentions no money

customer_email = "URGENT! My card was charged $450 twice for the annual subscription. Refund the duplicate today."

resp_json = client.models.generate_content(
    model=MODEL,
    contents=f"Triage this customer email:\n{customer_email}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",   # the answer must be JSON…
        response_schema=TicketTriage,            # …that matches this schema
        temperature=0.0,                         # extraction → no creativity
        thinking_config=types.ThinkingConfig(thinking_level="minimal"),
    ),
)

print("=== 1.3a RAW JSON FROM THE API ===")
print(resp_json.text)

triage = resp_json.parsed                        # 1.3b already a validated Python object
print("\n=== 1.3b PARSED OBJECT (ready for a database) ===")
print(f"category: {triage.category} | urgent: {triage.urgent} | amount_usd: {triage.amount_usd}")

## Step 1.4: Errors are normal — handle them in code

The client from Step 0 already retries **temporary** problems (429 busy, 5xx server errors, timeouts). **Your** mistakes — a wrong key (400) or a wrong model name (404) — are not retried: fix them.
Catch API errors with `errors.APIError` instead of letting the script crash:

In [ ]:
from google.genai import errors

try:
    client.models.generate_content(model="gemini-model-that-does-not-exist", contents="Ping")
except errors.APIError as e:
    print(f"Caught API error → HTTP {e.code} ({e.status})")   # expected: 404 NOT_FOUND
    print(str(e)[:200])